### Imports

In [2]:
import json
import numpy as np
import faiss
import requests

from sentence_transformers import SentenceTransformer

### Load FAISS + metadata

In [4]:
INDEX_PATH = "../data/embeddings/faiss.index"
META_PATH = "../data/embeddings/metadata.json"

index = faiss.read_index(INDEX_PATH)

with open(META_PATH, "r", encoding="utf-8") as f:
    chunks = json.load(f)

print(f"Loaded {len(chunks)} chunks")

Loaded 5932 chunks


### Load embedding model

In [6]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

### Retrieval

In [8]:
def search(query, k=20):
    q_emb = model.encode([query]).astype("float32")
    distances, indices = index.search(q_emb, k)
    return [chunks[i] for i in indices[0]]

### Deduplication

In [10]:
def unique_results(results, max_results=5):
    seen = set()
    unique = []

    for r in results:
        text = r["text"]

        if text[:200] not in seen:
            unique.append(r)
            seen.add(text[:200])

        if len(unique) >= max_results:
            break

    return unique

### Build context

In [12]:
def build_context(results):
    return "\n\n".join([r["text"] for r in results])

### Prompt (VERY IMPORTANT for local models)

In [14]:
def build_prompt(query, context):
    return f"""
You are an assistant that answers questions about my Machine Learning portfolio.

IMPORTANT:
- Use ONLY the provided context
- If the answer is not in the context, say "I don't know"
- Answer clearly and shortly

Context:
{context}

Question:
{query}

Answer:
"""

### Call local LLM (Ollama)

In [16]:
def ask_llm(prompt):
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "llama3",
            "prompt": prompt,
            "stream": False
        }
    )
    
    return response.json()["response"]

### Full RAG pipeline

In [18]:
def rag(query):
    # 1. Retrieve
    results = search(query, k=20)
    
    # 2. Deduplicate
    results = unique_results(results, max_results=5)
    
    # 3. Build context
    context = build_context(results)
    
    # 4. Prompt
    prompt = build_prompt(query, context)
    
    # 5. LLM
    answer = ask_llm(prompt)
    
    return answer, results

### Test

In [75]:
query = "What model is used in churn prediction?"

answer, results = rag(query)

print("ANSWER:\n")
print(answer)

ANSWER:

According to the context, several machine learning models were trained and evaluated for churn prediction, including:

* XGBoost
* LightGBM
* CatBoost
* KNN

No specific model was chosen as the best performer, so it's not clear which one is used in the churn prediction service.


In [73]:
query = "What is the recall of the best churn prediction model, that is used in prod?"

answer, results = rag(query)

print("ANSWER:\n")
print(answer)

ANSWER:

96%


In [35]:
# 3. Traffic system architecture
query = "Describe the architecture of the traffic sign detection system."
answer, results = rag(query)
print("\n--- Q3 ---")
print("ANSWER:\n", answer)


--- Q3 ---
ANSWER:
 The architecture of the traffic sign detection system is a **two-stage hybrid architecture**: **YOLO (detection) → ResNet (classification)**.


In [37]:
# 4. Improvement vs baseline
query = "How is the traffic sign detection system better than a baseline YOLO model?"
answer, results = rag(query)
print("\n--- Q4 ---")
print("ANSWER:\n", answer)


--- Q4 ---
ANSWER:
 The traffic sign detection system with the two-stage architecture (YOLO + ResNet) is not directly compared to a baseline YOLO model in the provided context. Therefore, I don't know.


In [39]:
# 5. Engineering challenges
query = "What engineering challenges were solved in the traffic sign detection project?"
answer, results = rag(query)
print("\n--- Q5 ---")
print("ANSWER:\n", answer)


--- Q5 ---
ANSWER:
 The engineering challenges that were solved in the traffic sign detection project include:

* Training stability: The use of a suitable optimizer (AdamW) and hyperparameters (learning rate, batch size) ensured smooth convergence without spikes or divergence.
* Overfitting prevention: Data augmentation techniques (`Blur`, `MedianBlur`, `CLAHE`, `ToGray`) prevented overfitting.

Note that these challenges are not explicitly stated in the text as "engineering challenges", but rather can be inferred from the provided information.


In [41]:
# 6. Full pipeline
query = "Describe the full pipeline of the churn prediction system."
answer, results = rag(query)
print("\n--- Q6 ---")
print("ANSWER:\n", answer)


--- Q6 ---
ANSWER:
 Based on the provided context, here is a description of the full pipeline:

1. **Data Loading & Exploration**: Load and explore the dataset to understand its structure and potential issues.
2. **Preprocessing**: Encode categorical features (one-hot / target encoding / ordinal where appropriate) and proceed with bivariate analysis (e.g., month-to-month) to analyze churn behavior.
3. **Modeling**: Train and evaluate multiple machine learning models for churn prediction, including boosting models (XGBoost, LightGBM, CatBoost), KNN, and others.
4. **Algorithm Comparison**: Compare the performance of each model using recall (churn detection) and other metrics to identify the best performer.

This pipeline is used to build a production-ready ML service that predicts customer churn for telecom companies, enabling proactive retention before customers leave.


In [43]:
# 7. Backend tech
query = "What technologies were used to build the churn prediction API?"
answer, results = rag(query)
print("\n--- Q7 ---")
print("ANSWER:\n", answer)


--- Q7 ---
ANSWER:
 According to the context, the technologies used to build the churn prediction API are:

* FastAPI
* CatBoost
* Streamlit
* Docker


In [77]:
# 7. Backend tech
query = "What technologies were used to build the sign detection API?"
answer, results = rag(query)
print("\n--- Q8 ---")
print("ANSWER:\n", answer)


--- Q7 ---
ANSWER:
 Docker, Caddy HTTPS, WebSocket.


In [45]:
# 8. Skills
query = "What machine learning skills does this portfolio demonstrate?"
answer, results = rag(query)
print("\n--- Q9 ---")
print("ANSWER:\n", answer)


--- Q8 ---
ANSWER:
 According to the context, this portfolio demonstrates practical data science and machine learning skills, specifically showcasing growth from fundamentals to advanced applied analytics.


In [47]:
# 9. Deployment
query = "How is the project deployed?"
answer, results = rag(query)
print("\n--- Q10 ---")
print("ANSWER:\n", answer)


--- Q9 ---
ANSWER:
 According to the context, the project is deployment-ready and can be loaded and used in production environments. Additionally, it mentions "deployment (Docker)" as a key design decision, indicating that Docker containers are used for deploying the project.


In [49]:
# 10. Production readiness
query = "What makes this portfolio production-ready?"
answer, results = rag(query)
print("\n--- Q11 ---")
print("ANSWER:\n", answer)


--- Q10 ---
ANSWER:
 Based on the provided context, what makes this portfolio production-ready is:

* **Practical problem-solving**: The portfolio focuses on solving real-world problems.
* **Clean analytical workflows**: It demonstrates a focus on reproducible and clean analytical workflows.
* **Reproducible Python-based data science**: The portfolio uses Python as the primary language.

Additionally, there are mentions of deploying models (Docker) and having CI/CD for ML services, which suggests that the portfolio is not only focused on theoretical knowledge but also on practical implementation.


In [51]:
# 11. Why recall
query = "Why was recall chosen as the main metric for churn prediction?"
answer, results = rag(query)
print("\n--- Q12 ---")
print("ANSWER:\n", answer)


--- Q11 ---
ANSWER:
 The primary metric chosen for churn prediction is Recall because the cost of missing a churned customer (false negative) is high - lost revenue, acquisition cost, and negative word-of-mouth.


In [53]:
# 12. Summary
query = "Summarize the churn project in 2 sentences."
answer, results = rag(query)
print("\n--- Q13 ---")
print("ANSWER:\n", answer)


--- Q12 ---
ANSWER:
 The churn project aims to predict customer churn based on various categorical features, including customer ID, gender, partner, dependents, and phone service. The dataset shows moderate class imbalance, with 73.5% of customers staying and 26.5% churning, and exhibits meaningful patterns in numerical features and distributions in categorical features.


In [55]:
# 13. Edge case
query = "Is there any information about the churn model algorithm?"
answer, results = rag(query)
print("\n--- Q14 ---")
print("ANSWER:\n", answer)


--- Q13 ---
ANSWER:
 Yes, according to the context, it says: "Boosting models (XGBoost, LightGBM, CatBoost) achieved the highest recall (>0.88), detecting most churn cases."


In [59]:
query = "Who is portfolio author?"
answer, results = rag(query)
print("\n--- Q14 ---")
print("ANSWER:\n", answer)


--- Q13 ---
ANSWER:
 Ivan


In [69]:
query = "What's the documentation about? Tell me how the Churn prediction service works?"
answer, results = rag(query)
print("\n--- Q15 ---")
print("ANSWER:\n", answer)


--- Q13 ---
ANSWER:
 The documentation is about the Churn Prediction Service, a production-ready machine learning service that predicts customer churn for telecom companies. The service provides real-time and batch predictions via REST API, an interactive dashboard for analytics, manual labeling system (ground truth collection), and a retraining pipeline for continuous improvement.

As for how it works, the documentation mentions the following:

* It's built with FastAPI as the backend
* Uses CatBoost as the ML model
* Has a frontend using Streamlit
* Stores data in SQLite
* Is containerized with Docker and Docker Compose
* Authenticates users through API Keys and Role-Based Access Control (RBAC)

It also mentions that the service provides real-time and batch predictions, an interactive dashboard for analytics, manual labeling system for ground truth collection, and a retraining pipeline for continuous improvement.


In [ ]:
query = "Is strong partfolio for junior?"
answer, results = rag(query)
print("\n--- Q15 ---")
print("ANSWER:\n", answer)